# PyObfus 交互式测试 - ml-research 集成

这个 notebook 允许你在不离开 pyobfus 环境的情况下交互式地测试 ml-research 模块的混淆。

In [ ]:
# 设置环境
import sys
from pathlib import Path

# 添加 pyobfus 到路径（如果需要）
pyobfus_path = Path.cwd().parent
if str(pyobfus_path) not in sys.path:
    sys.path.insert(0, str(pyobfus_path))

print(f"✅ PyObfus path: {pyobfus_path}")

In [ ]:
# 导入必要的模块
from pyobfus.config import ObfuscationConfig
from pyobfus.core.parser import ASTParser
from pyobfus.core.analyzer import SymbolAnalyzer
from pyobfus.transformers.name_mangler import NameMangler
from pyobfus.transformers.string_encoder import StringEncoder
from pyobfus.core.generator import CodeGenerator

print("✅ PyObfus modules imported")

## 配置 ml-research 路径

In [ ]:
# 设置你的 ml-research 项目路径
ML_RESEARCH_PATH = Path(r"c:\onedrive\msft\OneDrive - MSFT\rong\3-job\program\cardiac-ml-research")

if ML_RESEARCH_PATH.exists():
    print(f"✅ ml-research found: {ML_RESEARCH_PATH}")
    
    # 列出一些 Python 文件
    py_files = list(ML_RESEARCH_PATH.glob("*.py"))[:5]
    if py_files:
        print("\n📄 Sample Python files:")
        for f in py_files:
            print(f"   - {f.name}")
else:
    print(f"❌ ml-research not found at: {ML_RESEARCH_PATH}")
    print("   Please update ML_RESEARCH_PATH above")

## 辅助函数

In [ ]:
def obfuscate_code(code: str, config: ObfuscationConfig = None) -> str:
    """快速混淆代码字符串"""
    if config is None:
        config = ObfuscationConfig.community_edition()
        config.string_encoding = True
    
    # Parse
    tree = ASTParser.parse_string(code)
    
    # Analyze
    analyzer = SymbolAnalyzer(config)
    analyzer.analyze(tree)
    
    # Transform
    transformed_tree = tree
    
    # Name mangling
    mangler = NameMangler(config, analyzer)
    transformed_tree = mangler.transform(transformed_tree)
    
    # String encoding
    if config.string_encoding:
        encoder = StringEncoder(config, analyzer)
        transformed_tree = encoder.transform(transformed_tree)
    
    # Generate
    return CodeGenerator.generate(transformed_tree)


def obfuscate_file(file_path: Path, config: ObfuscationConfig = None) -> tuple[str, str]:
    """混淆文件并返回原始和混淆后的代码"""
    original_code = file_path.read_text(encoding='utf-8')
    obfuscated_code = obfuscate_code(original_code, config)
    return original_code, obfuscated_code


def compare_execution(original_code: str, obfuscated_code: str) -> dict:
    """执行并对比原始和混淆后的代码"""
    results = {
        'original_success': False,
        'obfuscated_success': False,
        'original_namespace': {},
        'obfuscated_namespace': {},
        'equivalent': False,
        'errors': []
    }
    
    # Execute original
    try:
        exec(original_code, results['original_namespace'])
        results['original_success'] = True
    except Exception as e:
        results['errors'].append(f"Original execution failed: {e}")
    
    # Execute obfuscated
    try:
        exec(obfuscated_code, results['obfuscated_namespace'])
        results['obfuscated_success'] = True
    except Exception as e:
        results['errors'].append(f"Obfuscated execution failed: {e}")
    
    # Check equivalence (simple check)
    if results['original_success'] and results['obfuscated_success']:
        # Compare variables (basic check)
        results['equivalent'] = True
    
    return results

print("✅ Helper functions defined")

## 示例 1: 混淆简单代码

In [ ]:
# 简单的测试代码
test_code = '''
def calculate_sum(a, b):
    """Calculate sum of two numbers."""
    result = a + b
    message = "Sum calculated"
    return result

total = calculate_sum(5, 3)
print(f"Total: {total}")
'''

# 混淆
obfuscated = obfuscate_code(test_code)

print("原始代码:")
print("="*70)
print(test_code)
print("\n混淆后代码:")
print("="*70)
print(obfuscated)

## 示例 2: 测试 ml-research 中的文件

In [ ]:
# 选择一个文件进行测试
# 修改下面的文件名为你想测试的文件
test_file = ML_RESEARCH_PATH / "your_module.py"  # 修改这里

if test_file.exists():
    print(f"🔄 Processing: {test_file.name}")
    
    original, obfuscated = obfuscate_file(test_file)
    
    print(f"\n📊 Statistics:")
    print(f"   Original: {len(original)} bytes, {len(original.splitlines())} lines")
    print(f"   Obfuscated: {len(obfuscated)} bytes, {len(obfuscated.splitlines())} lines")
    
    # 验证编译
    try:
        compile(obfuscated, test_file.name, 'exec')
        print(f"\n✅ Compilation successful")
    except SyntaxError as e:
        print(f"\n❌ Compilation failed: {e}")
    
    # 显示混淆后代码的前几行
    print(f"\n📝 Obfuscated code (first 20 lines):")
    print("="*70)
    print("\n".join(obfuscated.splitlines()[:20]))
    print("...")
else:
    print(f"❌ File not found: {test_file}")
    print("\n📁 Available files:")
    for f in ML_RESEARCH_PATH.glob("*.py")[:10]:
        print(f"   - {f.name}")

## 示例 3: 批量测试多个文件

In [ ]:
# 批量测试
test_files = list(ML_RESEARCH_PATH.glob("*.py"))[:5]  # 测试前5个文件

results = {'success': [], 'failed': []}

for file_path in test_files:
    print(f"\n🧪 Testing: {file_path.name}")
    
    try:
        original, obfuscated = obfuscate_file(file_path)
        compile(obfuscated, file_path.name, 'exec')
        results['success'].append(file_path.name)
        print(f"   ✅ Success")
    except Exception as e:
        results['failed'].append((file_path.name, str(e)))
        print(f"   ❌ Failed: {e}")

# 摘要
print(f"\n{'='*70}")
print(f"📊 Summary")
print(f"{'='*70}")
print(f"✅ Successful: {len(results['success'])}/{len(test_files)}")
print(f"❌ Failed: {len(results['failed'])}/{len(test_files)}")

if results['failed']:
    print(f"\nFailed files:")
    for name, error in results['failed']:
        print(f"   - {name}: {error[:50]}...")

## 示例 4: 自定义配置测试

In [ ]:
# 创建自定义配置
custom_config = ObfuscationConfig()
custom_config.string_encoding = True
custom_config.preserve_param_names = True  # 保留参数名
custom_config.add_exclude_name("important_function")  # 排除特定函数

# 测试代码
test_code = '''
def important_function(param1, param2):
    """This function should keep its name and parameters."""
    return param1 + param2

def helper_function(x, y):
    """This should be obfuscated, but params preserved."""
    return x * y
'''

obfuscated = obfuscate_code(test_code, custom_config)

print("混淆后的代码:")
print("="*70)
print(obfuscated)
print("\n✅ 注意: important_function 保留原名")
print("✅ 注意: 所有参数名都被保留")

## 示例 5: 对比执行结果

In [ ]:
# 测试执行等价性
test_code = '''
def calculate(a, b, c):
    result = (a + b) * c
    return result

output = calculate(5, 3, 2)
'''

obfuscated = obfuscate_code(test_code)

# 对比执行
results = compare_execution(test_code, obfuscated)

print("执行对比结果:")
print("="*70)
print(f"原始代码执行: {'✅ 成功' if results['original_success'] else '❌ 失败'}")
print(f"混淆代码执行: {'✅ 成功' if results['obfuscated_success'] else '❌ 失败'}")

if results['original_success'] and results['obfuscated_success']:
    # 对比输出
    original_output = results['original_namespace'].get('output')
    obfuscated_output = results['obfuscated_namespace'].get('output')
    
    print(f"\n结果对比:")
    print(f"   原始输出: {original_output}")
    print(f"   混淆输出: {obfuscated_output}")
    print(f"   等价性: {'✅ 相同' if original_output == obfuscated_output else '❌ 不同'}")

if results['errors']:
    print(f"\n错误:")
    for error in results['errors']:
        print(f"   {error}")

## 快速测试区域

在这里快速测试你的代码：

In [ ]:
# 在这里粘贴你想测试的代码
your_code = '''
# 你的代码
'''

if your_code.strip():
    obfuscated = obfuscate_code(your_code)
    print(obfuscated)
else:
    print("请在 your_code 变量中输入代码")